In [1]:
from miners.deberta_classifier import DebertaClassifier

/home/ine/miniconda3/envs/bittensor/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ine/miniconda3/envs/bittensor/lib/python3.10/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [2]:
deberta_foundation_model_path = "../models/deberta-v3-large-hf-weights"
deberta_model_path = "../models/deberta-large-ls03-ctx1024.pth"
device = "cuda:0"

In [3]:
model = DebertaClassifier(foundation_model_path=deberta_foundation_model_path,
                        model_path=deberta_model_path,
                        device=device)

/home/ine/miniconda3/envs/bittensor/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:473: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
/home/ine/miniconda3/envs/bittensor/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/ine/miniconda3/envs/bittensor/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._py

In [5]:
import pandas as pd
import glob
import os

from datasets import load_dataset

# Use only 100,000 samples for faster training
SAMPLE_SIZE = 100  # Adjust this as needed

print(f"Loading dataset from HuggingFace...")
dataset = load_dataset("../datasets/ahmadreza13/data", split="train")
    
print(f"Full dataset size: {len(dataset):,} examples")

# SHUFFLE FIRST before sampling (critical!)
print("Shuffling dataset...")
dataset = dataset.shuffle(seed=42)
print(f"Sampling {SAMPLE_SIZE} examples...")
dataset_sampled = dataset.select(range(SAMPLE_SIZE))

Loading dataset from HuggingFace...


HF google storage unreachable. Downloading and preparing it from source
Generating train split: 3614247 examples [00:05, 669152.84 examples/s] 


Full dataset size: 3,614,247 examples
Shuffling dataset...
Sampling 100 examples...


In [7]:
input_data = dataset_sampled['data']

print(f"Loaded and shuffled {len(dataset_sampled)} rows from all parquet files.")
print(f"Test set size: {len(dataset_sampled)}")
print(f"Sample: {input_data[0][:100]}...")

Loaded and shuffled 100 rows from all parquet files.
Test set size: 100
Sample: Brzeźno, Toruń County

Brzeźno is a village in the administrative district of Gmina Lubicz, within T...


In [8]:
preds = model.predict_batch(input_data)
# preds = [[pred] * len(text.split()) for pred, text in zip(preds, input_data)]

You're using a DebertaV2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
/home/ine/projects/sn32/neurons/miners/deberta_classifier.py:40: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


In [10]:
# Convert predicted probabilities to binary labels using a threshold (e.g., 0.5)
import numpy as np
from sklearn.metrics import precision_score
true_labels = dataset_sampled['generated']

binary_preds = (np.array(preds) > 0.5).astype(int)
print('Binary predicted label unique values:', set(binary_preds))
precision = precision_score(true_labels, binary_preds, average='binary')
print(f"Precision: {precision:.4f}")

Binary predicted label unique values: {np.int64(0), np.int64(1)}
Precision: 0.3605


In [14]:
print(dataset_sampled[0], preds[0])


{'data': 'Brzeźno, Toruń County\n\nBrzeźno is a village in the administrative district of Gmina Lubicz, within Toruń County, Kuyavian-Pomeranian Voivodeship, in north-central Poland. It lies approximately north of Lubicz and north-east of Toruń.\n\nThe village has a population of 370.', 'generated': 0, 'model': 'wikipedia'} 0.9844
